In [ ]:
!pip install lightgbm -q

import pandas as pd, numpy as np, zipfile
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report
import joblib

from google.colab import files
uploaded = files.upload()   # select the .zip you downloaded from Kaggle

In [ ]:
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall("dataset")
    print(z.namelist())   # note the csv filename printed here

In [ ]:
import os
for root, dirs, filenames in os.walk("dataset"):
    for f in filenames:
        print(os.path.join(root, f))

In [ ]:
df = pd.read_csv("dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv")
print(df.shape)
df.head()

In [ ]:
print(df['diseases'].nunique(), "unique diseases")

counts = df['diseases'].value_counts()
valid_diseases = counts[counts >= 5].index
df = df[df['diseases'].isin(valid_diseases)].reset_index(drop=True)
print("After filtering:", df.shape, df['diseases'].nunique(), "diseases left")

In [ ]:
X = df.drop(columns=['diseases'])
y = df['diseases']

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

In [ ]:
mi = mutual_info_classif(X_train, y_train, discrete_features=True, random_state=42)
mi_series = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)

mi_series.head(20).plot(kind='barh', figsize=(8,6))
plt.title("Top 20 most informative symptoms")
plt.gca().invert_yaxis()
plt.show()

top_features = mi_series.head(150).index
X_train_sel = X_train[top_features]
X_test_sel = X_test[top_features]

In [ ]:
from sklearn.linear_model import LogisticRegression
print("imports work")

In [ ]:
import re

def clean_name(name):
    return re.sub(r'[^A-Za-z0-9_]+', '_', name)

X_train_sel.columns = [clean_name(c) for c in X_train_sel.columns]
X_test_sel.columns = [clean_name(c) for c in X_test_sel.columns]
top_features = X_train_sel.columns  # keep this updated to match cleaned names

In [ ]:
import warnings
import logging

# Suppress LightGBM's internal logger
logging.getLogger("lightgbm").setLevel(logging.ERROR)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=25, n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=20, max_depth=20, n_jobs=-1, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=30, learning_rate=0.05, num_leaves=64, n_jobs=-1, verbose=-1)
}

results = {}
for name, model in models.items():
    model.fit(X_train_sel, y_train)
    proba = model.predict_proba(X_test_sel)
    top1 = accuracy_score(y_test, proba.argmax(axis=1))
    top3 = top_k_accuracy_score(y_test, proba, k=3)
    results[name] = {"top1": top1, "top3": top3, "model": model}
    print(f"{name}: Top-1 = {top1:.4f} | Top-3 = {top3:.4f}")

In [ ]:
best_name = max(results, key=lambda k: results[k]['top3'])
best_model = results[best_name]['model']
print("Best model:", best_name)

y_pred = best_model.predict(X_test_sel)
report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T.sort_values('support', ascending=True)
report_df.head(15)

### Hyperparameter Tuning for LightGBM (Extended Search)

To further improve performance and get closer to your target accuracy, we will perform a more extensive hyperparameter search for the `LGBMClassifier` using `RandomizedSearchCV`. We'll increase the number of iterations to explore more combinations.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from lightgbm import LGBMClassifier # Added this import

# Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': randint(100, 1000),  # Number of boosting rounds (increased range)
    'learning_rate': uniform(0.01, 0.2), # Step size shrinkage
    'num_leaves': randint(20, 150),    # Max number of leaves in one tree (increased range)
    'max_depth': randint(5, 40),        # Max tree depth (increased range)
    'reg_alpha': uniform(0, 0.7),       # L1 regularization (increased range)
    'reg_lambda': uniform(0, 0.7),      # L2 regularization (increased range)
    'colsample_bytree': uniform(0.6, 0.4), # Subsample ratio of columns when constructing each tree
    'subsample': uniform(0.6, 0.4),      # Subsample ratio of the training instance
    'min_child_samples': randint(10, 150) # Minimum number of data needed in a child (leaf) (increased range)
}

lgbm = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

# Initialize RandomizedSearchCV with increased n_iter
rand_search = RandomizedSearchCV(
    lgbm,
    param_distributions=param_dist,
    n_iter=100,  # Increased number of parameter settings to sample
    cv=3,       # 3-fold cross-validation
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Fit RandomizedSearchCV to the training data
print("Starting Extended RandomizedSearchCV for LightGBM...")

# Check if X_train_sel and y_train are defined before fitting
if 'X_train_sel' not in locals() and 'X_train_sel' not in globals() or \
   'y_train' not in locals() and 'y_train' not in globals():
    print("\nError: X_train_sel or y_train are not defined. Please ensure preceding data preprocessing and feature selection cells have been executed.\n")
    raise NameError("X_train_sel or y_train not found. Please run previous cells.")

# Initialize 'results' dictionary if it's not defined, to prevent errors.
if 'results' not in locals() and 'results' not in globals():
    print("Warning: 'results' dictionary not found. Initializing an empty dictionary.")
    results = {}

rand_search.fit(X_train_sel, y_train)
print("Extended RandomizedSearchCV finished.")

# Get the best estimator
best_lgbm = rand_search.best_estimator_
print(f"Best LightGBM parameters: {rand_search.best_params_}")

# Evaluate the best LightGBM model
proba_tuned_lgbm = best_lgbm.predict_proba(X_test_sel)
top1_tuned_lgbm = accuracy_score(y_test, proba_tuned_lgbm.argmax(axis=1))
top3_tuned_lgbm = top_k_accuracy_score(y_test, proba_tuned_lgbm, k=3)

print(f"Tuned LightGBM: Top-1 = {top1_tuned_lgbm:.4f} | Top-3 = {top3_tuned_lgbm:.4f}")

# Update results with the tuned LightGBM model if it's better
if 'LightGBM' not in results or top1_tuned_lgbm > results['LightGBM']['top1']:
    results['LightGBM'] = {"top1": top1_tuned_lgbm, "top3": top3_tuned_lgbm, "model": best_lgbm}

# Re-determine the overall best model after tuning
best_name = max(results, key=lambda k: results[k]['top3'])
best_model = results[best_name]['model']
print("\nNew overall best model after tuning:", best_name)

In [ ]:
cv_scores = cross_val_score(best_model, X_train_sel, y_train, cv=5, n_jobs=-1)
print("CV mean accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
joblib.dump(best_model, "disease_model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(list(top_features), "selected_features.pkl")
files.download("disease_model.pkl")

In [ ]:
def predict_disease(symptom_list, top_n=3):
    input_vec = pd.Series(0, index=top_features)
    for s in symptom_list:
        if s in input_vec.index:
            input_vec[s] = 1
    proba = best_model.predict_proba(input_vec.values.reshape(1, -1))[0]
    top_idx = np.argsort(proba)[::-1][:top_n]
    return [(le.classes_[i], round(proba[i]*100, 2)) for i in top_idx]

predict_disease(['fever', 'cough', 'fatigue'])

In [ ]:
from sklearn.metrics import accuracy_score

train_acc = accuracy_score(y_train, best_model.predict(X_train_sel))
test_acc = accuracy_score(y_test, best_model.predict(X_test_sel))

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

text = widgets.Text(
    description='Symptoms:',
    placeholder='fever,cough,fatigue'
)

button = widgets.Button(description="Predict")
output = widgets.Output()

def on_button_click(b):
    with output:
        output.clear_output()
        symptom_list = [s.strip().lower() for s in text.value.split(",")]
        result = predict_disease(symptom_list)
        print("Top Predicted Diseases:")
        for disease, probability in result:
            print(f"{disease}: {probability:.2f}%")

button.on_click(on_button_click)

display(text, button, output)